In [1]:


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

# Load the dataset from the same folder as the notebook
diamonds = pd.read_csv("diamonds.csv")

# Set the required ordering for colour and clarity
colour_order = ["D", "E", "F", "G", "H", "I", "J"]
clarity_order = ["SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

# Use a clean style similar to the example
sns.set_theme(style="white")


# Interactive heatmap for average carat by colour and clarity
@widgets.interact(
    Origin=widgets.Dropdown(
        options=["Natural", "Lab", "All"],
        value="Natural",
        description="Origin"
    )
)
def plot_heatmap(Origin):
    # Filter based on dropdown selection
    filtered = diamonds.copy()
    if Origin == "Natural":
        filtered = filtered[filtered["type"] == "natural"]
    elif Origin == "Lab":
        filtered = filtered[filtered["type"] == "lab"]

    # Build the table of average carat values
    heatmap_data = (
        filtered.pivot_table(
            index="colour",
            columns="clarity",
            values="carat",
            aggfunc="mean"
        )
        .reindex(index=colour_order, columns=clarity_order)
    )

    # Draw the heatmap
    plt.figure(figsize=(8, 4.6))
    ax = sns.heatmap(
        heatmap_data,
        cmap="mako",
        vmin=0.5,
        vmax=1.4
    )

    # Add axis labels
    ax.set_xlabel("Clarity")
    ax.set_ylabel("Colour")

    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='Origin', options=('Natural', 'Lab', 'All'), value='Natural'), Outp…

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

names = pd.read_csv("names.csv")

# Keep the plotting style clean
sns.set_theme(style="white")

# Colours required for highlighted names
highlight_colours = ["#206095", "#004662", "#A8BD3A", "#27A0CC", "#118C7B", "#F66068"]

# Years shown in the chart
years = list(range(1996, 2021))

# Create the interactive controls
sex_toggle = widgets.ToggleButtons(
    options=[("Baby boy names", "boy"), ("Baby girl names", "girl")],
    value="boy"
)

name_box = widgets.Text(
    value="",
    description="Name:"
)

output = widgets.Output()


# Draw the chart based on the selected sex and typed names
def update_chart(sex, typed_names):
    output.clear_output(wait=True)

    with output:
        # Only keep the top 100 names from 2020 for the selected sex
        top_100_2020 = names[(names["year"] == 2020) & (names["sex"] == sex) & (names["rank"] <= 100)]
        valid_names = top_100_2020["name"].tolist()

        # Keep data for only those 2020 top 100 names across all years
        plot_data = names[(names["sex"] == sex) & (names["name"].isin(valid_names))].copy()

        # Parse the text input, make it case-insensitive, remove duplicates, max 6 names
        selected_names = []
        for item in typed_names.split():
            item = item.upper()
            if item in valid_names and item not in selected_names:
                selected_names.append(item)
            if len(selected_names) == 6:
                break

        fig, ax = plt.subplots(figsize=(8.2, 5.2))

        # Draw all top-100 lines in faded grey
        for baby_name in valid_names:
            name_data = plot_data[plot_data["name"] == baby_name][["year", "rank"]].copy()
            name_data = pd.DataFrame({"year": years}).merge(name_data, on="year", how="left")

            ax.plot(
                name_data["year"],
                name_data["rank"],
                color="#BFBFBF",
                alpha=0.18,
                linewidth=1.0,
                zorder=1
            )

        # Draw highlighted names with points
        legend_labels = []
        for i, baby_name in enumerate(selected_names):
            name_data = plot_data[plot_data["name"] == baby_name][["year", "rank"]].copy()
            name_data = pd.DataFrame({"year": years}).merge(name_data, on="year", how="left")

            ax.plot(
                name_data["year"],
                name_data["rank"],
                color=highlight_colours[i],
                linewidth=1.4,
                marker="o",
                markersize=3.2,
                zorder=3,
                label=baby_name.capitalize()
            )
            legend_labels.append(baby_name.capitalize())

        # Match the required styling
        ax.set_title("Popularity ranking (1 being the most popular)")
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticks(list(range(1996, 2021, 2)))
        ax.set_yticks([1] + list(range(100, 1001, 100)))
        ax.set_ylim(1000, 1)
        ax.grid(axis="y", color="#D9D9D9", linewidth=1)
        ax.grid(axis="x", visible=False)

        # Remove left, right and top borders
        ax.spines["left"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)

        # Only show the legend when at least one valid name is highlighted
        if legend_labels:
            ax.legend(loc="lower right", frameon=True)

        plt.tight_layout()
        plt.show()


# Connect the widgets to the chart
interactive_plot = widgets.interactive_output(
    update_chart,
    {"sex": sex_toggle, "typed_names": name_box}
)

# Display the controls and chart
display(widgets.VBox([sex_toggle, name_box, output]), interactive_plot)

Output()